# Component Modeling

This notebook benchmarks the structured component prediction workflow on the single-label vehicle complaint cases.

The goal is to compare simple baselines against a stronger CatBoost model on a strict time-aware split before we decide how much more complexity the task needs.

## 1. Setup and Ground Rules

We load the modeling libraries, define the split boundaries, and lock the structured feature lists.

In [ ]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    top_k_accuracy_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Robust pathing to directories and data
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'odi_component_model_cases.parquet'

# Model time and target definitions
TARGET_COL = 'component_group'
DATE_COL = 'ldate'
TRAIN_END = pd.Timestamp('2024-12-31')
VALID_END = pd.Timestamp('2025-12-31')
RANDOM_SEED = 42

# Constants
CAT_COLS = [
    'mfr_name',
    'maketxt',
    'modeltxt',
    'state',
    'cmpl_type',
    'drive_train',
    'fuel_sys',
    'fuel_type',
    'trans_type',
    'fire',
    'crash',
    'medical_attn',
    'vehicles_towed_yn',
    'police_rpt_yn',
    'orig_owner_yn',
    'anti_brakes_yn',
    'cruise_cont_yn'
]

NUM_COLS = [
    'yeartxt',
    'miles',
    'veh_speed',
    'injured',
    'deaths',
    'num_cyls',
    'lag_days_safe'
]

FLAG_COLS = [
    'miles_missing_flag',
    'veh_speed_missing_flag',
    'faildate_untrusted_flag',
    'flag_year_unknown',
    'flag_speed_high',
    'flag_miles_high'
]

FEATURE_COLS = CAT_COLS + NUM_COLS + FLAG_COLS
DATA_PATH

WindowsPath('c:/Users/davis/Documents/VCCode Repos/NHTSA-ODI-Complaint-Analytics/data/processed/odi_component_model_cases.parquet')

## 2. Load And Audit The Modeling Table

We load the collapsed single-label component case table that was created from `src/features/collapse_components.py` and do a quick audit of rows, dates, and target coverage to confirm setup.

In [ ]:
df = pd.read_parquet(DATA_PATH).copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

required_cols = FEATURE_COLS + [TARGET_COL, DATE_COL, 'odino']
missing_cols = [column for column in required_cols if column not in df.columns]
if missing_cols:
    missing_text = ', '.join(missing_cols)
    raise ValueError(f'Missing required columns: {missing_text}')

for column in CAT_COLS:
    df[column] = df[column].astype(object)
    df[column] = df[column].where(pd.notna(df[column]), np.nan)

for column in NUM_COLS + FLAG_COLS:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Get general overview of data for this model
overview = pd.DataFrame([{
    'rows': int(len(df)),
    'columns': int(len(df.columns)),
    'target_groups': int(df[TARGET_COL].nunique()),
    'date_min': df[DATE_COL].min(),
    'date_max': df[DATE_COL].max()
}])

# Get the number of rows per year
year_counts = (
    df[DATE_COL]
    .dt.year
    .value_counts()
    .sort_index()
    .rename_axis('ldate_year')
    .reset_index(name='rows')
)

# Get the number of rows per component group
target_counts = (
    df[TARGET_COL]
    .value_counts()
    .rename_axis('component_group')
    .reset_index(name='rows')
)

print("General model data overview:")
display(overview)

print("\nRow counts per year in collapsed data:")
display(year_counts)

print("\nRow counts per target component group:")
display(target_counts)

General model data overview:


,rows,columns,target_groups,date_min,date_max
0,252122,43,22,2020-01-01,2026-02-19



Row counts per year in collapsed data:


,ldate_year,rows
0,2020,39380
1,2021,33392
2,2022,36335
3,2023,42247
4,2024,45669
5,2025,48098
6,2026,7001



Row counts per target component group:


,component_group,rows
0,ENGINE / COOLING,52770
1,POWER TRAIN,31344
2,ELECTRICAL SYSTEM,28053
3,STEERING,21735
4,SERVICE BRAKES,20122
5,AIR BAGS,15781
6,FUEL / PROPULSION,15554
7,STRUCTURE,14596
8,VISIBILITY / WIPER,11729
9,EXTERIOR LIGHTING,7357


## 3. Build The Time-Aware Split

We split by complaint receipt date so validation uses later complaints the model has not seen.

This keeps the benchmark honest and leaves the partial `2026` slice untouched for future holdout testing.

In [5]:
train_df = df.loc[df[DATE_COL] <= TRAIN_END].copy()
valid_df = df.loc[(df[DATE_COL] > TRAIN_END) & (df[DATE_COL] <= VALID_END)].copy()
holdout_df = df.loc[df[DATE_COL] > VALID_END].copy()

split_summary = pd.DataFrame([
    {
        'split': 'train',
        'rows': int(len(train_df)),
        'cases': int(train_df['odino'].nunique()),
        'date_min': train_df[DATE_COL].min(),
        'date_max': train_df[DATE_COL].max(),
        'target_groups': int(train_df[TARGET_COL].nunique())
    },
    {
        'split': 'valid',
        'rows': int(len(valid_df)),
        'cases': int(valid_df['odino'].nunique()),
        'date_min': valid_df[DATE_COL].min(),
        'date_max': valid_df[DATE_COL].max(),
        'target_groups': int(valid_df[TARGET_COL].nunique())
    },
    {
        'split': 'holdout_2026',
        'rows': int(len(holdout_df)),
        'cases': int(holdout_df['odino'].nunique()),
        'date_min': holdout_df[DATE_COL].min(),
        'date_max': holdout_df[DATE_COL].max(),
        'target_groups': int(holdout_df[TARGET_COL].nunique())
    }
])

# Make sure no component group labels exist in the validation data that don't in the training data
unseen_valid_labels = sorted(set(valid_df[TARGET_COL]) - set(train_df[TARGET_COL]))

X_train = train_df[FEATURE_COLS].copy()
y_train = train_df[TARGET_COL].copy()
X_valid = valid_df[FEATURE_COLS].copy()
y_valid = valid_df[TARGET_COL].copy()

print("Summary of rows and targets per time split: ")
display(split_summary)

print('Unseen validation labels:', unseen_valid_labels)

Summary of rows and targets per time split: 


,split,rows,cases,date_min,date_max,target_groups
0,train,197023,197023,2020-01-01,2024-12-31,22
1,valid,48098,48098,2025-01-01,2025-12-31,22
2,holdout_2026,7001,7001,2026-01-01,2026-02-19,21


Unseen validation labels: []


## 4. Naive Baseline

Start with the simplest possible reference point: always predict the most common training label. This gives the stronger models a concrete floor to beat.

In [ ]:
train_target_counts = y_train.value_counts()
majority_label = train_target_counts.idxmax()
top3_labels = train_target_counts.head(3).index.tolist()

naive_pred = pd.Series(majority_label, index=y_valid.index)
naive_summary = pd.DataFrame([{
    'model': 'Most Frequent Label',
    'top_1_accuracy': round(accuracy_score(y_valid, naive_pred), 4),
    'macro_f1': round(f1_score(y_valid, naive_pred, average='macro'), 4),
    'top_3_accuracy': round(y_valid.isin(top3_labels).mean(), 4)
}])

print("Most frequent training label:", majority_label)
print("Top 3 training labels:", top3_labels)
print("\nNaive baseline model evaluation:")
display(naive_summary)

Most frequent training label: ENGINE / COOLING
Top 3 training labels: ['ENGINE / COOLING', 'POWER TRAIN', 'ELECTRICAL SYSTEM']
/nNaive baseline model evaluation:


,model,top_1_accuracy,macro_f1,top_3_accuracy
0,Most Frequent Label,0.2598,0.0187,0.5295


## 5. Logistic Regression Baseline

This baseline uses one-hot encoding and a linear model on the structured feature set. This is simple, interpretable, and gives us a clean comparison point before tree boosting.

In [8]:
# Imputation and encoding pipeline for categorical features
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Imputation and scaling pipeline for numeric features
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Column transformer to divert columns to correct pipeline
prep = ColumnTransformer([
    ('cat', cat_pipe, CAT_COLS),
    ('num', num_pipe, NUM_COLS + FLAG_COLS)
])

# Logistic regression pipeline to prevent leakage
logit_model = Pipeline([
        ('prep', prep),
        ('model', LogisticRegression(
            solver='saga',
            class_weight='balanced',
            max_iter=300,
            random_state=RANDOM_SEED
        ))
])

start = perf_counter()
logit_model.fit(X_train, y_train)
fit_seconds = perf_counter() - start
fit_seconds

c:\Users\davis\Documents\VCCode Repos\NHTSA-ODI-Complaint-Analytics\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


196.76685640000505

### 5.1 Logistic Overall Metrics

After fitting, we score the logistic model on both train and validation.

In [ ]:
def score_split(split_name, y_true, pred, proba, classes):
    return {
        'split': split_name,
        'rows': int(len(y_true)),
        'top_1_accuracy': round(accuracy_score(y_true, pred), 4),
        'macro_f1': round(f1_score(y_true, pred, average='macro'), 4),
        'top_3_accuracy': round(top_k_accuracy_score(y_true, proba, labels=classes, k=3), 4)
    }

train_pred = logit_model.predict(X_train)
train_proba = logit_model.predict_proba(X_train)
valid_pred = logit_model.predict(X_valid)
valid_proba = logit_model.predict_proba(X_valid)
classes = logit_model.named_steps['model'].classes_

baseline_results = pd.DataFrame([
    score_split('train', y_train, train_pred, train_proba, classes),
    score_split('valid_2025', y_valid, valid_pred, valid_proba, classes)
])

print(f"Fit time (seconds): {fit_seconds:.2f}")
print("\nLogistic Regression baseline results:")
display(baseline_results)

Fit time (seconds): 196.77

Logistic Regression baseline results:


,split,rows,top_1_accuracy,macro_f1,top_3_accuracy
0,train,197023,0.2608,0.2136,0.5090
1,valid_2025,48098,0.1899,0.1461,0.4158


### 5.2 Logistic Class Level Review

Overall metrics can hide weak classes, so we break the logistic baseline down by component group.

That gives us an early view of which labels are easy or hard with a linear model.

In [12]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_valid,
    valid_pred,
    labels=classes,
    zero_division=0
)

class_results = pd.DataFrame(
    {
        'component_group': classes,
        'support': support,
        'precision': np.round(precision, 4),
        'recall': np.round(recall, 4),
        'f1': np.round(f1, 4)
    }
).sort_values(['support', 'f1'], ascending=[False, False]).reset_index(drop=True)

print("Logistic Regression baseline evaluation by class:")
display(class_results)

Logistic Regression baseline evaluation by class:


,component_group,support,precision,recall,f1
0,ENGINE / COOLING,12494,0.6969,0.1667,0.2691
1,POWER TRAIN,7196,0.5097,0.2018,0.2891
2,ELECTRICAL SYSTEM,5776,0.3659,0.0959,0.1520
3,SERVICE BRAKES,3307,0.4480,0.2422,0.3144
4,STEERING,3260,0.3563,0.2893,0.3193
5,STRUCTURE,2888,0.2418,0.1101,0.1513
6,FUEL / PROPULSION,2703,0.0979,0.0559,0.0711
7,AIR BAGS,2016,0.3209,0.2937,0.3067
8,VISIBILITY / WIPER,1520,0.1379,0.1368,0.1374
9,BACK OVER PREVENTION,1304,0.1324,0.5782,0.2154


### 5.3 Logistic Confusion Review

This confusion matrix view focuses on the largest classes in the training data.

It makes the biggest routing mistakes easier to see than the overall metrics alone.

In [14]:
focus_groups = train_df[TARGET_COL].value_counts().head(12).index.tolist()
confusion_counts = pd.crosstab(
    pd.Categorical(y_valid, categories=focus_groups),
    pd.Categorical(valid_pred, categories=focus_groups),
    dropna=False
)
confusion_share = confusion_counts.div(
    confusion_counts.sum(axis=1).replace(0, np.nan),
    axis=0
).round(3)

print("Focused confusion counts for the 12 largest training groups")
display(confusion_counts)

print("Focused row-normalized confusion shares")
display(confusion_share)

Focused confusion counts for the 12 largest training groups


C:\Users\davis\AppData\Local\Temp\ipykernel_27152\1485684454.py:3: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  pd.Categorical(y_valid, categories=focus_groups),
C:\Users\davis\AppData\Local\Temp\ipykernel_27152\1485684454.py:4: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  pd.Categorical(valid_pred, categories=focus_groups),


col_0,ENGINE / COOLING,POWER TRAIN,ELECTRICAL SYSTEM,STEERING,SERVICE BRAKES,AIR BAGS,FUEL / PROPULSION,STRUCTURE,VISIBILITY / WIPER,EXTERIOR LIGHTING,SUSPENSION,FORWARD COLLISION AVOIDANCE
row_0,,,,,,,,,,,,
NaN,59,58,77,55,71,145,73,102,132,78,53,273
ENGINE / COOLING,2083,765,342,377,362,227,323,241,242,1005,94,722
POWER TRAIN,216,1452,127,219,231,217,137,194,129,290,58,268
ELECTRICAL SYSTEM,154,142,554,224,90,196,302,101,214,235,79,415
STEERING,93,133,83,943,84,74,161,104,36,74,92,447
SERVICE BRAKES,79,91,60,220,801,108,123,61,155,94,50,152
AIR BAGS,24,13,38,41,21,592,83,20,82,87,34,93
FUEL / PROPULSION,102,70,118,132,41,100,151,82,116,96,127,286
STRUCTURE,86,36,32,309,33,63,85,318,75,132,63,129


Focused row-normalized confusion shares


col_0,ENGINE / COOLING,POWER TRAIN,ELECTRICAL SYSTEM,STEERING,SERVICE BRAKES,AIR BAGS,FUEL / PROPULSION,STRUCTURE,VISIBILITY / WIPER,EXTERIOR LIGHTING,SUSPENSION,FORWARD COLLISION AVOIDANCE
row_0,,,,,,,,,,,,
NaN,0.050,0.049,0.065,0.047,0.060,0.123,0.062,0.087,0.112,0.066,0.045,0.232
ENGINE / COOLING,0.307,0.113,0.050,0.056,0.053,0.033,0.048,0.036,0.036,0.148,0.014,0.106
POWER TRAIN,0.061,0.410,0.036,0.062,0.065,0.061,0.039,0.055,0.036,0.082,0.016,0.076
ELECTRICAL SYSTEM,0.057,0.052,0.205,0.083,0.033,0.072,0.112,0.037,0.079,0.087,0.029,0.153
STEERING,0.040,0.057,0.036,0.406,0.036,0.032,0.069,0.045,0.015,0.032,0.040,0.192
SERVICE BRAKES,0.040,0.046,0.030,0.110,0.402,0.054,0.062,0.031,0.078,0.047,0.025,0.076
AIR BAGS,0.021,0.012,0.034,0.036,0.019,0.525,0.074,0.018,0.073,0.077,0.030,0.082
FUEL / PROPULSION,0.072,0.049,0.083,0.093,0.029,0.070,0.106,0.058,0.082,0.068,0.089,0.201
STRUCTURE,0.063,0.026,0.024,0.227,0.024,0.046,0.062,0.234,0.055,0.097,0.046,0.095


## 6. CatBoost Benchmark

CatBoost is the main structured benchmark because it handles native categoricals, missing values, and nonlinear interactions well. It's a strong tree model that uses boosting to classify predictions into multiple potential target features.

This section uses the same time-aware split and feature set so the comparison against the earlier baselines stays fair.

In [ ]:
def prep_catboost_frame(frame):
    frame = frame.copy()
    for column in CAT_COLS:
        frame[column] = frame[column].astype('string').fillna('__MISSING__')
    for column in NUM_COLS + FLAG_COLS:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    return frame


X_train_cat = prep_catboost_frame(X_train)
X_valid_cat = prep_catboost_frame(X_valid)

cat_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    iterations=100,
    learning_rate=0.12,
    depth=5,
    l2_leaf_reg=6,
    random_seed=RANDOM_SEED,
    allow_writing_files=False,
    verbose=25
)

start = perf_counter()
cat_model.fit(
    X_train_cat,
    y_train,
    cat_features=CAT_COLS,
    eval_set=(X_valid_cat, y_valid),
    use_best_model=True,
    early_stopping_rounds=25
)
cat_fit_seconds = perf_counter() - start
cat_fit_seconds

### 6.1 CatBoost Overall Metrics

This cell scores the CatBoost benchmark on both train and validation and compares it against the earlier baselines.

The question here is whether the stronger structured model gives a meaningful predictive increase, not just a tiny incremental gain.

In [ ]:
cat_train_pred = pd.Series(cat_model.predict(X_train_cat).ravel(), index=y_train.index)
cat_train_proba = cat_model.predict_proba(X_train_cat)
cat_valid_pred = pd.Series(cat_model.predict(X_valid_cat).ravel(), index=y_valid.index)
cat_valid_proba = cat_model.predict_proba(X_valid_cat)
cat_classes = cat_model.classes_

catboost_results = pd.DataFrame([
    score_split('train', y_train, cat_train_pred, cat_train_proba, cat_classes),
    score_split('valid_2025', y_valid, cat_valid_pred, cat_valid_proba, cat_classes)
])

naive_compare = pd.DataFrame([
    {
        'model': 'Most Frequent Label',
        'split': 'valid_2025',
        'rows': int(len(y_valid)),
        'top_1_accuracy': round(accuracy_score(y_valid, naive_pred), 4),
        'macro_f1': round(f1_score(y_valid, naive_pred, average='macro'), 4),
        'top_3_accuracy': round(y_valid.isin(top3_labels).mean(), 4)
    }
])

logit_compare = baseline_results.copy()
logit_compare.insert(0, 'model', 'Logistic Regression')
catboost_compare = catboost_results.copy()
catboost_compare.insert(0, 'model', 'CatBoost')
model_compare = pd.concat([naive_compare, logit_compare, catboost_compare], ignore_index=True)

print(f"CatBoost fit time (seconds): {cat_fit_seconds:.2f}")
print("\nCatBoost results:")
display(catboost_results)

print("\nModel comparison:")
display(model_compare)

### 6.2 CatBoost Class-Level Review

We break the CatBoost results down by component group to see where the benchmark is strong and where it is still weak.

If we later add text work, it should target the classes that structured features still struggle to separate.

In [ ]:
cat_precision, cat_recall, cat_f1, cat_support = precision_recall_fscore_support(
    y_valid,
    cat_valid_pred,
    labels=cat_classes,
    zero_division=0
)

cat_class_results = pd.DataFrame(
    {
        'component_group': cat_classes,
        'support': cat_support,
        'precision': np.round(cat_precision, 4),
        'recall': np.round(cat_recall, 4),
        'f1': np.round(cat_f1, 4)
    }
).sort_values(['support', 'f1'], ascending=[False, False]).reset_index(drop=True)

print("CatBoost validation evaluation by class:")
display(cat_class_results)

### 6.3 CatBoost Confusion And Importance Review

This last diagnostic block shows the largest class confusions and the feature-importance profile.

Together they show what the structured benchmark is learning and where the remaining ambiguity likely lives.

In [ ]:
cat_confusion_counts = pd.crosstab(
    pd.Categorical(y_valid, categories=focus_groups),
    pd.Categorical(cat_valid_pred, categories=focus_groups),
    dropna=False
)
cat_confusion_share = cat_confusion_counts.div(
    cat_confusion_counts.sum(axis=1).replace(0, np.nan),
    axis=0
).round(3)

cat_importance = pd.DataFrame(
    {
        'feature': FEATURE_COLS,
        'importance': np.round(cat_model.get_feature_importance(), 4)
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

print("CatBoost focused confusion counts for the 12 largest training groups")
display(cat_confusion_counts)

print("Focused row-normalized confusion shares")
display(cat_confusion_share)

print("CatBoost feature importance")
display(cat_importance.head(20))

## 7. Next Steps

At this point the notebook has done its job: define the split, benchmark the simple baselines, and establish the structured CatBoost reference.

The fully set model parameters are in `src/modeling/component_catboost.py` after fine tuning the hyperparameters using Optuna in Colab (due to the powerful GPU runtimes it has available). The script use to use a fine search of Optuna trials is in `src\modeling\tune_component_catboost.py`.